# RAGAS Single-Turn Evaluation — End-to-End (Azure OpenAI)

A beginner-friendly, ready-to-run notebook for evaluating **single-turn** LLM / RAG
responses with [RAGAS](https://docs.ragas.io).

A *single-turn* evaluation looks at **one question and its one answer** at a time
(no back-and-forth conversation). For each example RAGAS can use up to four pieces
of information and gives you a score per metric.

**What's inside**
1. Install & configure RAGAS with Azure OpenAI
2. Understand the single-turn data format
3. The built-in ("defined") metrics and what each one measures
4. **Two ways to run an evaluation:**
   - **Method A — Editable samples:** type your own examples directly in the notebook (great for a quick test)
   - **Method B — Excel upload:** score a whole spreadsheet of questions & answers at once
5. **Custom metrics** with **Aspect Critic** (define your own pass/fail criteria)

**Prerequisites**
- Python 3.10+ (Google Colab works out of the box)
- An Azure OpenAI resource: endpoint, API key, deployment name, API version

> 💡 New here? Just run the cells top to bottom. Every section has a short note before the code.


## 1. Install dependencies

Run this once per session. If the packages are already installed you can skip it.


In [ ]:
%pip install \
    ragas==0.3.9 \
    langchain==0.3.27 \
    langchain-openai==0.3.28 \
    langchain-community==0.3.27 \
    langchain-huggingface==0.1.2 \
    openai==1.99.5 \
    datasets==4.0.0 \
    pandas==2.2.2 \
    sentence-transformers==3.4.1 \
    nest_asyncio==1.6.0 \
    openpyxl==3.1.5
print('Done. Skip this cell next time if packages are already installed.')

## 2. Imports

Core libraries used throughout the notebook. (Metric-specific imports live in their
own sections so it's clear what each part needs.)


In [ ]:
import os
import pandas as pd

from ragas import SingleTurnSample, EvaluationDataset, evaluate

# nest_asyncio lets RAGAS run its async calls inside a notebook / Colab cell.
import nest_asyncio
nest_asyncio.apply()

print('Imports ready')

## 3. Configure Azure OpenAI credentials

Fill in the four values below with your own Azure OpenAI details. The next cell
just confirms they were picked up (it never prints the key itself).


In [ ]:
import os

# 👇 Replace these with your Azure OpenAI values
os.environ["AZURE_OPENAI_API_KEY"]     = "YOUR_AZURE_OPENAI_API_KEY"
os.environ["AZURE_OPENAI_API_VERSION"] = "YOUR_AZURE_OPENAI_API_VERSION"
os.environ["AZURE_OPENAI_ENDPOINT"]    = "YOUR_AZURE_OPENAI_ENDPOINT"
os.environ["AZURE_DEPLOYMENT_NAME"]    = "YOUR_AZURE_DEPLOYMENT_NAME" 

In [ ]:
api_key     = os.environ.get('AZURE_OPENAI_API_KEY')     or os.environ.get('OPENAI_API_KEY')
api_version = os.environ.get('AZURE_OPENAI_API_VERSION') or os.environ.get('OPENAI_API_VERSION')
endpoint    = os.environ.get('AZURE_OPENAI_ENDPOINT')    or os.environ.get('OPENAI_API_BASE')
deployment  = os.environ.get('AZURE_DEPLOYMENT_NAME')    or os.environ.get('DEPLOYMENT_NAME')

print('API key present?', bool(api_key))
print('Endpoint:', endpoint)
print('Deployment:', deployment)

## 4. Initialize the LLM & embeddings for RAGAS

Most RAGAS metrics use an **LLM as a judge**, and a few (similarity-based) also need
an **embedding model**. Here we wrap Azure OpenAI as the judge LLM and a small local
HuggingFace model for embeddings.

> You may see `DeprecationWarning`s about the wrapper classes — they are safe to
> ignore on RAGAS 0.3.x and the code works as-is.


In [ ]:
try:
    # AzureChatOpenAI now lives in langchain_openai (the langchain_community path is deprecated).
    from langchain_openai import AzureChatOpenAI
    from langchain_huggingface import HuggingFaceEmbeddings
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    use_langchain = True
except Exception as e:
    print('langchain_openai / langchain_huggingface not available. Error:', e)
    use_langchain = False

if use_langchain:
    # LangChain AzureChatOpenAI instance (Azure-style parameters)
    llm_client = AzureChatOpenAI(
        azure_deployment=deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version=api_version,
        temperature=1,
    )
    wrapped_llm = LangchainLLMWrapper(llm_client, bypass_temperature=True)
    print('Wrapped LangChain AzureChatOpenAI for RAGAS')
else:
    # Fallback: wrap your own Azure client with LangchainLLMWrapper
    wrapped_llm = None
    print('No LLM wrapper created; set up your preferred Azure client and wrap it.')

# Embeddings used by similarity-based metrics (downloads a small model on first run)
evaluator_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)
print('Embeddings ready')

## 5. The single-turn data format

Every example you evaluate is a `SingleTurnSample` built from up to four fields.
Not every metric needs all four — the table below shows what each field means.

| Field | Meaning | Example |
|---|---|---|
| `user_input` | the question / prompt | *"How many sick leave days are allowed each year?"* |
| `reference` | the ground-truth / expected answer | *"7 days of sick leave per year."* |
| `response` | the answer your model actually produced | *"Employees get 7 sick leave days each year."* |
| `retrieved_contexts` | the passage(s) your RAG system retrieved (a list) | *["Employees are entitled to 7 days of sick leave annually..."]* |

We use these exact field names everywhere below.


## 6. Built-in ("defined") metrics

These are the ready-made metrics this notebook uses. Each needs only certain fields,
so if your data is missing a field, that metric simply can't run.

| Metric | What it measures | Needs |
|---|---|---|
| **AnswerCorrectness** | How well the response matches the reference (facts + meaning) | `response`, `reference` |
| **Faithfulness** | Is the response actually supported by the retrieved contexts? (no hallucination) | `response`, `retrieved_contexts` |
| **ContextRecall** | Did retrieval bring back the info needed to answer correctly? | `user_input`, `reference`, `retrieved_contexts` |
| **ContextPrecision** | Are the retrieved contexts relevant (and well-ranked) for the question? | `user_input`, `reference`, `retrieved_contexts` |
| **NoiseSensitivity** | How much does irrelevant retrieved context throw the answer off? | all four |
| **AnswerSimilarity** | Embedding similarity between response and reference | `response`, `reference` |

The cell below creates these metric instances once so every section can reuse them.


In [ ]:
from ragas.metrics import (
    AnswerCorrectness,
    AnswerSimilarity,
    Faithfulness,
    ContextRecall,
    ContextPrecision,
    NoiseSensitivity,
)

# Similarity metric needs the embedding model
answer_similarity = AnswerSimilarity(embeddings=evaluator_embeddings)

# The full built-in metric set (used when all four fields are present, e.g. Method A)
metrics = [
    AnswerCorrectness(llm=wrapped_llm, answer_similarity=answer_similarity),
    Faithfulness(llm=wrapped_llm),
    ContextRecall(llm=wrapped_llm),
    ContextPrecision(llm=wrapped_llm),
    NoiseSensitivity(llm=wrapped_llm),
]

# Each metric paired with the fields it requires. Used to auto-skip metrics
# when a column is missing (see Method B) so a run never crashes.
METRIC_REQUIREMENTS = [
    ("answer_correctness", lambda: AnswerCorrectness(llm=wrapped_llm, answer_similarity=answer_similarity), {"response", "reference"}),
    ("faithfulness",       lambda: Faithfulness(llm=wrapped_llm),     {"response", "retrieved_contexts"}),
    ("context_recall",     lambda: ContextRecall(llm=wrapped_llm),    {"user_input", "reference", "retrieved_contexts"}),
    ("context_precision",  lambda: ContextPrecision(llm=wrapped_llm), {"user_input", "reference", "retrieved_contexts"}),
    ("noise_sensitivity",  lambda: NoiseSensitivity(llm=wrapped_llm), {"user_input", "response", "reference", "retrieved_contexts"}),
    ("answer_similarity",  lambda: answer_similarity,                 {"response", "reference"}),
]

def select_applicable_metrics(present_fields):
    # Return only the metrics whose required fields are all present.
    chosen, skipped = [], []
    for name, factory, required in METRIC_REQUIREMENTS:
        if required.issubset(present_fields):
            chosen.append(factory())
        else:
            skipped.append((name, required - present_fields))
    if skipped:
        print('Skipping metrics (missing fields):')
        for name, miss in skipped:
            print(f'   - {name}  (needs: {", ".join(sorted(miss))})')
    print('Running metrics:', [getattr(m, "name", type(m).__name__) for m in chosen])
    return chosen

print('Metrics ready.')

## 7. Method A — Evaluate your own editable samples

Best for a quick check. The dictionaries below are **placeholders** — edit the text
to match an example you care about, then run the cells. Everything stays in the
notebook; no file needed.


### 7.1 Placeholder examples (edit these)

Two ready-to-edit cases: one where the answer is good, and one where it's clearly
wrong. Change the strings to your own content.


In [ ]:
# A well-answered case
sample_case = {
    "input": "How much maternity leave are female employees entitled to?",
    "expected_output": "Female employees are entitled to 26 weeks of paid maternity leave as per the Maternity Benefit Act.",
    "actual_output": "Female employees receive 26 weeks of paid maternity leave, which can start up to 8 weeks before delivery.",
    "retrieval_context": [
        "Maternity leave provides 26 weeks of paid time off for female employees.",
        "Up to 8 weeks can be taken before the expected date of delivery.",
    ],
}

# A mismatched case: the answer does not match the question/reference
mismatched_case = {
    "input": "What should employees wear on casual Fridays?",
    "expected_output": "Employees may wear clean jeans and simple t-shirts without rips or offensive prints.",
    "actual_output": "Employees can take one extra leave day on Fridays as part of work-life balance initiatives.",
    "retrieval_context": [
        "Casual Fridays allow jeans and casual shoes, provided clothing is clean and appropriate.",
        "Slogans or political prints are not permitted on t-shirts.",
    ],
}

### 7.2 Build samples and run the built-in metrics

Here we turn three editable examples into `SingleTurnSample`s, bundle them into an
`EvaluationDataset`, and score them with the full built-in metric set from Section 6.


In [ ]:
# Edit these three samples freely — add or remove as you like.
sample1 = SingleTurnSample(
    user_input="What can employees wear on casual Fridays?",
    retrieved_contexts=["Clean jeans and plain t-shirts are allowed; no rips or offensive prints."],
    response="Employees may wear neat jeans and simple t-shirts on casual Fridays.",
    reference="Clean jeans and plain t-shirts without rips or prints.",
)
sample2 = SingleTurnSample(
    user_input="How many sick leave days are allowed each year?",
    retrieved_contexts=["Employees are entitled to 7 days of sick leave annually with a medical note if absent over 3 days."],
    response="Employees get 7 sick leave days each year, with a doctor's note if more than 3 days.",
    reference="7 days of sick leave per year.",
)
sample3 = SingleTurnSample(
    user_input="Can employees book travel independently?",
    retrieved_contexts=["Travel must be booked through approved vendors unless written approval is given."],
    response="Employees can book travel only with prior written approval from the travel coordinator.",
    reference="Only with prior written approval from the travel coordinator or finance.",
)

dataset = EvaluationDataset(samples=[sample1, sample2, sample3])

response = evaluate(dataset=dataset, metrics=metrics, llm=wrapped_llm, embeddings=evaluator_embeddings)
print(response.to_pandas().to_string())

### 7.3 (Optional) Score a single placeholder case

You can also evaluate just one of the dictionaries above. This reuses `sample_case`
and a smaller metric list.


In [ ]:
single_df = pd.DataFrame([{
    'user_input': sample_case['input'],
    'response': sample_case['actual_output'],
    'retrieved_contexts': sample_case['retrieval_context'],
    'reference': sample_case['expected_output'],
}])
single_dataset = EvaluationDataset.from_pandas(single_df)

single_metrics = [
    AnswerCorrectness(llm=wrapped_llm, answer_similarity=answer_similarity),
    Faithfulness(llm=wrapped_llm),
    ContextRecall(llm=wrapped_llm),
]

try:
    eval_result = evaluate(dataset=single_dataset, metrics=single_metrics,
                           llm=wrapped_llm, embeddings=evaluator_embeddings)
    print('Evaluate() returned:')
    print(eval_result)
except Exception as e:
    print('evaluate(...) failed:', e)
    print('Ensure the ragas.evaluate signature matches your installed version.')

## 8. Method B — Evaluate from an Excel file (Colab upload)

Best for scoring many rows at once. Each **row** = one example; each **column** maps
to a RAGAS field.

**Expected columns** (generic names — case/spacing don't matter):

| Excel column | maps to RAGAS field |
|---|---|
| `Question` | `user_input` |
| `Ground_Truth` | `reference` |
| `Answer` | `response` |
| `Contexts` | `retrieved_contexts` |

Notes for beginners:
- **Missing a column?** No problem — it's auto-detected and any metric that needs it
  is skipped (the run won't crash).
- **Multiple passages** in one `Contexts` cell? Separate them with `||` or new lines.


### 8.1 Upload your Excel file

Runs the Colab upload widget and reads the first sheet. Pick your `.xlsx` when prompted.


In [ ]:
from google.colab import files

uploaded = files.upload()  # choose your .xlsx file
excel_filename = next(iter(uploaded))            # first uploaded file
excel_df = pd.read_excel(excel_filename)         # reads the first/default sheet

print(f'Loaded "{excel_filename}" with {len(excel_df)} row(s).')
print('Columns found:', list(excel_df.columns))
excel_df.head()

### 8.2 Map columns and build the dataset (handles missing columns)

This detects which of your columns match each RAGAS field, splits multi-passage
context cells, and skips blank cells — so partial files still work.


In [ ]:
# Accepted header spellings for each RAGAS field (add your own if needed).
COLUMN_ALIASES = {
    "user_input":         ["question", "user_input", "input", "query", "prompt"],
    "reference":          ["ground_truth", "groundtruth", "reference", "expected_output", "expected", "gold_answer"],
    "response":           ["answer", "response", "actual_output", "output", "prediction", "model_answer"],
    "retrieved_contexts": ["contexts", "context", "retrieved_contexts", "retrieval_context", "retrieved_context"],
}
CONTEXT_DELIMITERS = ["||", "\n", ";"]   # how multiple passages in one cell may be separated

def _norm(name):
    return str(name).strip().lower().replace(" ", "_")

def build_field_map(columns):
    norm_cols = {_norm(c): c for c in columns}
    field_map = {}
    for field, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            if alias in norm_cols:
                field_map[field] = norm_cols[alias]
                break
    return field_map

def _is_blank(value):
    return value is None or (isinstance(value, float) and pd.isna(value)) or str(value).strip() == ""

def split_contexts(value):
    if _is_blank(value):
        return None
    text = str(value).strip()
    for delim in CONTEXT_DELIMITERS:
        if delim in text:
            parts = [p.strip() for p in text.split(delim) if p.strip()]
            return parts or None
    return [text]

def excel_to_dataset(df):
    field_map = build_field_map(df.columns)
    present = set(field_map.keys())

    print('Detected column -> RAGAS field:')
    for field, col in field_map.items():
        print(f'   {col!r}  ->  {field}')
    missing = set(COLUMN_ALIASES) - present
    if missing:
        print('\n⚠️  Columns not found (dependent metrics will be skipped):', ', '.join(sorted(missing)))

    samples = []
    for _, row in df.iterrows():
        kwargs = {}
        for field, col in field_map.items():
            value = row.get(col)
            if field == "retrieved_contexts":
                ctx = split_contexts(value)
                if ctx is not None:
                    kwargs[field] = ctx
            elif not _is_blank(value):
                kwargs[field] = str(value).strip()
        if kwargs:                      # skip completely empty rows
            samples.append(SingleTurnSample(**kwargs))

    print(f'\nBuilt {len(samples)} sample(s) from {len(df)} row(s).')
    return EvaluationDataset(samples=samples), present

excel_dataset, present_fields = excel_to_dataset(excel_df)

### 8.3 Run the evaluation

Only the metrics whose required fields are present will run.


In [ ]:
excel_metrics = select_applicable_metrics(present_fields)

excel_results = evaluate(
    dataset=excel_dataset,
    metrics=excel_metrics,
    llm=wrapped_llm,
    embeddings=evaluator_embeddings,
)

excel_scores_df = excel_results.to_pandas()
print(excel_scores_df.to_string())

# Optional: save the scores back to an Excel file you can download
excel_scores_df.to_excel('ragas_results.xlsx', index=False)
print('\nSaved scores to ragas_results.xlsx')

## 9. Custom metrics with Aspect Critic

Sometimes you want to check something the built-in metrics don't cover — e.g.
*"is the answer polite?"* or *"is it concise?"*. **Aspect Critic** lets you describe
a yes/no criterion in plain English, and the LLM judges it (returns **1 = pass**,
**0 = fail**).

Below are five custom criteria — adapt the `definition` text to whatever you need.


In [ ]:
from ragas.metrics import AspectCritic
from ragas.dataset_schema import SingleTurnSample

answer_correctness_critic = AspectCritic(
    name="Answer Correctness",
    definition="Does the answer correctly convey the same meaning as the reference answer?",
    llm=wrapped_llm,
)

fluency = AspectCritic(
    name="Fluency",
    definition="Is the answer grammatically correct and easy to understand?",
    llm=wrapped_llm,
)

completeness = AspectCritic(
    name="Completeness",
    definition="Does the answer fully address the question using the provided context?",
    llm=wrapped_llm,
)

conciseness = AspectCritic(
    name="Conciseness",
    definition="Is the answer to the point, without unnecessary or repeated information?",
    llm=wrapped_llm,
)

groundedness = AspectCritic(
    name="Groundedness",
    definition="Is every claim in the answer supported by the retrieved context, with nothing made up?",
    llm=wrapped_llm,
)

custom_metrics = [answer_correctness_critic, fluency, completeness, conciseness, groundedness]
print('Custom metrics ready:', [m.name for m in custom_metrics])

### 9.1 Run custom metrics on a single sample

Uses the `mismatched_case` from Section 7 so you can see how a wrong answer scores.


In [ ]:
sample = SingleTurnSample(
    user_input=mismatched_case['input'],
    retrieved_contexts=mismatched_case['retrieval_context'],
    response=mismatched_case['actual_output'],
    reference=mismatched_case['expected_output'],
)

async def run_custom_metrics_scores_only():
    print('Sample:')
    print(f'  User Input: {sample.user_input}')
    print(f'  Response:   {sample.response}')
    print(f'  Reference:  {sample.reference}')
    print('\n  Metric Scores (1 = pass, 0 = fail):')
    for metric in custom_metrics:
        score = await metric.single_turn_ascore(sample)
        print(f'    {metric.name}: {score}')

await run_custom_metrics_scores_only()

### 9.2 (Optional) Run custom metrics across a whole dataset

Aspect Critic metrics also work with `evaluate()`, so you can score every row of the
sample dataset (or swap in `excel_dataset`) at once.


In [ ]:
custom_results = evaluate(
    dataset=dataset,            # from Method A; use excel_dataset for the Excel rows
    metrics=custom_metrics,
    llm=wrapped_llm,
)
print(custom_results.to_pandas().to_string())

## 10. Recap & next steps

You now have a single-turn evaluation notebook that can:
- score **your own typed examples** (Method A), and
- score a **whole Excel sheet** (Method B), skipping any metric whose column is missing,
- using both **built-in metrics** and **custom Aspect Critic** criteria.

**Where to go next**
- Add more rows to your Excel file and re-run Section 8.
- Write new Aspect Critic definitions in Section 9 for criteria specific to your use case.
- Compare scores across model versions by changing the Azure deployment in Section 3.

📚 RAGAS docs: https://docs.ragas.io
